# Phase 5 — Food Type Classifier (EfficientNet-B0 on Food-101)

**Goal**: Add real-food-type recognition on top of the Phase 4 (YOLO + MiDaS + MLP) pipeline.  
Even when YOLO sees "unknown dish", this classifier will identify the actual food category  
("pizza", "sushi", "pad_thai", etc.) by looking directly at the image crop.

### Why this matters
- Phase 4 MLP predicts calories from **geometry only** (mask area + depth stats) — it can't tell pizza from salad.  
- This Phase 5 model adds a **food-type label** to every prediction, making the UI genuinely useful for real-world photo uploads.

### Pipeline after Phase 5
```
Input image
    ↓
YOLOv8-seg ──────────────────── food mask + segmentation
    ↓                               ↓
EfficientNet-B0 (this notebook)    MiDaS depth map
(food-type label, top-5)               ↓
                              9 geometric features
                                       ↓
                                  NutritionMLP
                                       ↓
                          calories / fat / protein / carbs ± uncertainty
```

### 🗂 Recommended Datasets (best for real-time food photos)

| Dataset | Categories | Images | Best for |
|---|---|---|---|
| **Food-101** ← *used here* | 101 | 101 K | Western meals, cafeteria food |
| **iFood-2019** (Kaggle competition) | 251 | 118 K | Fine-grained classification |
| **UECFOOD-256** | 256 | ~30 K | Japanese + Asian cuisine |
| **Food2K** | 2 000 | 1M+ | Maximum variety (needs big GPU) |
| **VFN / VIREO Food-172** | 172 | 110 K | Chinese food |
| **Open Images V7 food** | ~80 | millions | Real-world diversity |

**Recommendation for your use-case (real-time photos with many food types):**  
Start with **Food-101** (quick to train, 101 classes cover most common dishes).  
Then stack **iFood-2019** labels for 251 classes if you need more granularity.  
For Asian cuisines add **UECFOOD-256**. Combine all three with a unified label mapping for the best real-world accuracy.


## Cell 1 — Install / Import Required Libraries

In [ ]:
# ── Kaggle env check ──────────────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# timm gives us EfficientNet-B0 with pretrained ImageNet weights
try:
    import timm
except ImportError:
    pip('timm')

import os, json, random, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.datasets import Food101
import timm
from sklearn.metrics import top_k_accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {device}')
print(f'torchvision {torchvision.__version__}  |  timm {timm.__version__}')


## Cell 2 — Configuration & Hyperparameters

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# On Kaggle the output directory is /kaggle/working/
DATA_ROOT  = '/kaggle/working/food101_data'   # Food-101 will be downloaded here
OUTPUT_DIR = '/kaggle/working'
os.makedirs(DATA_ROOT, exist_ok=True)

FOOD_CLF_CKPT  = os.path.join(OUTPUT_DIR, 'best_food_classifier.pt')
FOOD_CLF_LABELS= os.path.join(OUTPUT_DIR, 'food101_labels.json')   # idx → class name

# ── Model / training config ───────────────────────────────────────────────────
CFG = dict(
    model_name    = 'efficientnet_b0',      # timm model id; ~5.3M params, fast on Kaggle T4
    img_size      = 224,
    batch_size    = 64,
    num_epochs    = 20,                     # Increase to 30-40 for a competition submission
    base_lr       = 3e-4,                   # head LR
    finetune_lr   = 5e-5,                   # backbone LR (10× lower)
    weight_decay  = 1e-4,
    warmup_epochs = 2,
    label_smooth  = 0.1,
    num_workers   = 2,
    num_classes   = 101,                    # Food-101
    # Mixup alpha — set 0.0 to disable
    mixup_alpha   = 0.4,
)

print('Config:')
for k, v in CFG.items():
    print(f'  {k:20s}: {v}')


## Cell 3 — Load & Explore Food-101

Food-101 ships in `torchvision.datasets.Food101`.  
First run downloads ~5 GB via HTTP — on Kaggle use the pre-staged dataset below instead if available.  
Classes: apple_pie, baby_back_ribs, baklava, … pizza, ramen, sushi, tacos, … (101 total).

In [ ]:
# ── Download Food-101 (run once) ──────────────────────────────────────────────
# torchvision downloads the official 5 GB tar archive from vision.ee.ethz.ch
# On Kaggle add: Datasets → food101 (kmader/food41) and point DATA_ROOT there.

_minimal_transform = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])

print('Downloading / reading Food-101 train split …')
train_raw = Food101(root=DATA_ROOT, split='train', download=True, transform=_minimal_transform)
test_raw  = Food101(root=DATA_ROOT, split='test',  download=True, transform=_minimal_transform)

# Class names — use the dataset's own mapping
CLASS_NAMES = train_raw.classes          # list[str], 101 entries, sorted alphabetically
NUM_CLASSES  = len(CLASS_NAMES)
print(f'Classes : {NUM_CLASSES}')
print(f'Train   : {len(train_raw):,} images')
print(f'Test    : {len(test_raw):,}  images')
print(f'First 10: {CLASS_NAMES[:10]}')

# Save label mapping for use in app.py
with open(FOOD_CLF_LABELS, 'w') as f:
    json.dump({str(i): name for i, name in enumerate(CLASS_NAMES)}, f, indent=2)
print(f'Saved label map → {FOOD_CLF_LABELS}')


In [ ]:
# ── Visualise sample images per class (first 20 classes) ─────────────────────
N_SHOW  = 20
N_EACH  = 3
fig, axes = plt.subplots(N_SHOW, N_EACH, figsize=(N_EACH * 2, N_SHOW * 2))

# index images by class
from collections import defaultdict
class_to_idx_list = defaultdict(list)
for idx, (_, label) in enumerate(train_raw):
    class_to_idx_list[label].append(idx)
    if all(len(v) >= N_EACH for v in class_to_idx_list.values()) and len(class_to_idx_list) >= N_SHOW:
        break

for row, cid in enumerate(sorted(class_to_idx_list)[:N_SHOW]):
    for col, img_idx in enumerate(class_to_idx_list[cid][:N_EACH]):
        img_tensor, _ = train_raw[img_idx]
        img_np = img_tensor.permute(1, 2, 0).numpy()
        axes[row, col].imshow(img_np)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(CLASS_NAMES[cid].replace('_', '\n'), fontsize=7, rotation=0, ha='right', va='center')

plt.suptitle('Food-101 — Sample Images (first 20 classes, 3 each)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Class distribution (should be balanced: 750 train, 250 test per class)
counts = [0] * NUM_CLASSES
for _, lbl in train_raw:
    counts[lbl] += 1
print(f'Train images per class — min: {min(counts)}  max: {max(counts)}  (expected 750 each)')


## Cell 4 — Data Preprocessing & Augmentation

Strong augmentation is critical because real-world food photos:
- Are taken from any angle (not always top-down)
- Have varying lighting / backgrounds
- May be partially cropped or occluded

We use RandAugment + Mixup to maximise generalisation.

In [ ]:
IMG_SIZE   = CFG['img_size']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Training transforms (heavy augmentation for real-world generalisation) ─────
train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.75, 1.33)),   # crop variation
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(p=0.1),                                           # rare overhead shots
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    T.RandomRotation(30),
    T.RandAugment(num_ops=2, magnitude=9),                                 # policy augment
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.15)),                           # hide small patches
])

# ── Validation transforms (deterministic) ─────────────────────────────────────
val_transform = T.Compose([
    T.Resize(int(IMG_SIZE * 1.143)),   # resize to 256 if IMG_SIZE=224
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Rebuild datasets with proper transforms ────────────────────────────────────
train_ds = Food101(root=DATA_ROOT, split='train', download=False, transform=train_transform)
val_ds   = Food101(root=DATA_ROOT, split='test',  download=False, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'] * 2, shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

print(f'Train batches: {len(train_loader)}  (batch_size={CFG["batch_size"]})')
print(f'Val   batches: {len(val_loader)}')

# ── Visualise train augmentation on one sample ────────────────────────────────
_raw_img, _lbl = train_raw[0]
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
axes[0].imshow(_raw_img.permute(1, 2, 0).numpy())
axes[0].set_title(f'Original\n{CLASS_NAMES[_lbl]}', fontsize=7)
axes[0].axis('off')
for i in range(1, 8):
    aug = train_transform(Image.fromarray(((_raw_img.permute(1,2,0).numpy() * 255).astype('uint8'))))
    img_disp = aug * torch.tensor(IMAGENET_STD).view(3,1,1) + torch.tensor(IMAGENET_MEAN).view(3,1,1)
    axes[i].imshow(img_disp.permute(1,2,0).clamp(0,1).numpy())
    axes[i].set_title(f'Aug {i}', fontsize=7)
    axes[i].axis('off')
plt.suptitle('Augmentation variety on a single image', fontsize=10)
plt.tight_layout()
plt.show()


## Cell 5 — Build EfficientNet-B0 Classifier

EfficientNet-B0 is the lightest EfficientNet (~5.3 M params).  
It achieves ~85 % top-1 on ImageNet and trains in < 20 min per epoch on a Kaggle T4.  
We replace the final classifier head with a two-layer dropout head for 101 classes.

In [ ]:
class FoodClassifier(nn.Module):
    """EfficientNet-B0 fine-tuned for food-type classification.

    Architecture
    ────────────
    backbone: EfficientNet-B0 (pretrained on ImageNet-1k, all layers unfrozen)
    head    : Dropout(0.3) → Linear(1280, 512) → ReLU → BN → Dropout(0.2) → Linear(512, num_classes)

    Using two dropout layers + batch norm dramatically reduces overfitting on the
    75-image-per-class Food-101 training split.
    """

    def __init__(self, num_classes: int = 101, dropout: float = 0.3, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'],
            pretrained=pretrained,
            num_classes=0,          # remove original classifier head
            global_pool='avg',      # global average pool → (B, 1280) vector
        )
        feat_dim = self.backbone.num_features   # 1280 for B0

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)    # (B, 1280)
        return self.head(feats)     # (B, num_classes) logits

    def get_embedding(self, x):
        """Return 1280-dim feature embedding (useful for downstream tasks)."""
        return self.backbone(x)


model = FoodClassifier(num_classes=NUM_CLASSES, pretrained=True).to(device)

# ── Parameter count ────────────────────────────────────────────────────────────
total_params    = sum(p.numel() for p in model.parameters())
trainable_params= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

# ── Quick forward-pass sanity check ───────────────────────────────────────────
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'Output shape: {out.shape}  (expected [2, {NUM_CLASSES}])')


## Cell 6 — Training Loop (with Mixup + Label Smoothing + Cosine LR)

In [ ]:
# ── Skip training if checkpoint already exists ─────────────────────────────────
if os.path.isfile(FOOD_CLF_CKPT):
    print(f'Checkpoint exists at {FOOD_CLF_CKPT} — skipping training.')
    print('Delete the file and re-run this cell to retrain.')
else:
    # ── Loss with label smoothing ──────────────────────────────────────────────
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])

    # ── Differential learning rates: backbone 10× lower than head ─────────────
    optimizer = optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': CFG['finetune_lr']},
        {'params': model.head.parameters(),     'lr': CFG['base_lr']},
    ], weight_decay=CFG['weight_decay'])

    # ── Cosine annealing with linear warmup ────────────────────────────────────
    def lr_lambda_warmup(epoch):
        if epoch < CFG['warmup_epochs']:
            return (epoch + 1) / CFG['warmup_epochs']
        return 1.0

    warmup_sched  = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda_warmup)
    cosine_sched  = CosineAnnealingLR(optimizer, T_max=CFG['num_epochs'] - CFG['warmup_epochs'], eta_min=1e-6)

    # ── Mixup helper ───────────────────────────────────────────────────────────
    def mixup_batch(x, y, alpha=0.4):
        if alpha <= 0:
            return x, y, y, 1.0
        lam = np.random.beta(alpha, alpha)
        idx = torch.randperm(x.size(0), device=x.device)
        x_mix = lam * x + (1 - lam) * x[idx]
        return x_mix, y, y[idx], lam

    def mixup_loss(criterion, pred, y_a, y_b, lam):
        return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

    # ── Training ───────────────────────────────────────────────────────────────
    history = {'train_loss': [], 'val_loss': [], 'val_top1': [], 'val_top5': []}
    best_val_acc = 0.0

    for epoch in range(CFG['num_epochs']):
        t0 = time.time()

        # ── Train ──────────────────────────────────────────────────────────────
        model.train()
        running_loss = correct = total = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            x_mix, y_a, y_b, lam = mixup_batch(batch_x, batch_y, alpha=CFG['mixup_alpha'])

            optimizer.zero_grad()
            logits = model(x_mix)
            loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * batch_x.size(0)
            # Top-1 accuracy on unmixed labels for rough training metric
            preds  = logits.argmax(1)
            correct += (lam * (preds == y_a).float() + (1-lam) * (preds == y_b).float()).sum().item()
            total  += batch_x.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        # ── Validate ───────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0; all_logits = []; all_targets = []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits  = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_x.size(0)
                all_logits.append(logits.cpu())
                all_targets.append(batch_y.cpu())
        val_loss /= len(val_ds)

        all_logits  = torch.cat(all_logits).numpy()
        all_targets = torch.cat(all_targets).numpy()
        val_top1 = (all_logits.argmax(1) == all_targets).mean()
        val_top5 = top_k_accuracy_score(all_targets, all_logits, k=5)

        # ── LR schedule ───────────────────────────────────────────────────────
        if epoch < CFG['warmup_epochs']:
            warmup_sched.step()
        else:
            cosine_sched.step()

        # ── Checkpoint ────────────────────────────────────────────────────────
        if val_top1 > best_val_acc:
            best_val_acc = val_top1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_top1': val_top1,
                'val_top5': val_top5,
                'class_names': CLASS_NAMES,
                'cfg': CFG,
            }, FOOD_CLF_CKPT)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_top1'].append(val_top1)
        history['val_top5'].append(val_top5)

        elapsed = time.time() - t0
        lr_now  = optimizer.param_groups[1]['lr']
        print(f'Epoch {epoch+1:3d}/{CFG["num_epochs"]}  '
              f'loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | '
              f'top1: {val_top1*100:.1f}% top5: {val_top5*100:.1f}%  '
              f'lr: {lr_now:.2e}  [{elapsed:.0f}s]')

    print(f'\n✓  Best val top-1: {best_val_acc*100:.2f}%   checkpoint → {FOOD_CLF_CKPT}')


## Cell 7 — Evaluate Model Performance

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt = torch.load(FOOD_CLF_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded epoch {ckpt["epoch"]+1}  |  val_top1: {ckpt["val_top1"]*100:.2f}%  '
      f'val_top5: {ckpt["val_top5"]*100:.2f}%')

# ── Plot training curves (only when history dict is available) ────────────────
if 'history' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history['train_loss'], label='train')
    axes[0].plot(history['val_loss'],   label='val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Cross-Entropy Loss'); axes[0].legend()

    axes[1].plot([v*100 for v in history['val_top1']], label='Top-1 %')
    axes[1].plot([v*100 for v in history['val_top5']], label='Top-5 %')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Validation Accuracy'); axes[1].legend()
    plt.tight_layout(); plt.show()

# ── Per-class accuracy — show top-10 and bottom-10 ────────────────────────────
model.eval()
class_correct = torch.zeros(NUM_CLASSES)
class_total   = torch.zeros(NUM_CLASSES)

with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        preds = model(batch_x).argmax(1)
        for c in range(NUM_CLASSES):
            mask = batch_y == c
            class_correct[c] += (preds[mask] == c).sum().cpu()
            class_total[c]   += mask.sum().cpu()

per_class_acc = (class_correct / class_total.clamp(min=1)).numpy()
sorted_idx = np.argsort(per_class_acc)

print('\nTop-10 easiest classes:')
for i in sorted_idx[-10:][::-1]:
    print(f'  {CLASS_NAMES[i]:30s}  {per_class_acc[i]*100:.1f}%')

print('\nBottom-10 hardest classes:')
for i in sorted_idx[:10]:
    print(f'  {CLASS_NAMES[i]:30s}  {per_class_acc[i]*100:.1f}%')

print(f'\nOverall Top-1 Accuracy: {per_class_acc.mean()*100:.2f}%')


## Cell 8 — Top-5 Prediction Display

Show the model's top-5 predictions with confidence bars — this is exactly what the Gradio app will display.

In [ ]:
import torch.nn.functional as F

def predict_top5(pil_image: Image.Image, model: nn.Module, class_names: list):
    """Return list of (class_name, probability) sorted by probability desc."""
    model.eval()
    x = val_transform(pil_image.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x).squeeze(0)
    probs  = F.softmax(logits, dim=0).cpu().numpy()
    top5_i = probs.argsort()[-5:][::-1]
    return [(class_names[i].replace('_', ' ').title(), float(probs[i])) for i in top5_i]


def show_prediction(pil_image: Image.Image, title: str = ''):
    preds = predict_top5(pil_image, model, CLASS_NAMES)
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 4))

    ax_img.imshow(pil_image.resize((224, 224)))
    ax_img.set_title(title or 'Input image', fontsize=10)
    ax_img.axis('off')

    names  = [p[0] for p in preds][::-1]
    scores = [p[1]*100 for p in preds][::-1]
    colors = ['#2196F3' if i == 4 else '#90CAF9' for i in range(5)]
    ax_bar.barh(names, scores, color=colors)
    ax_bar.set_xlabel('Confidence (%)')
    ax_bar.set_title('Top-5 Predictions')
    for i, (name, score) in enumerate(zip(names, scores)):
        ax_bar.text(score + 0.5, i, f'{score:.1f}%', va='center', fontsize=9)
    ax_bar.set_xlim(0, 105)

    plt.tight_layout()
    plt.show()
    print('Predictions:', preds)


# ── Show predictions on 6 random validation images ────────────────────────────
sample_indices = random.sample(range(len(val_ds)), 6)
for si in sample_indices:
    # Load raw PIL image (without transform applied)
    img_path = val_ds._image_files[si]
    pil_img  = Image.open(str(img_path)).convert('RGB')
    true_label = CLASS_NAMES[val_ds._labels[si]].replace('_', ' ').title()
    show_prediction(pil_img, title=f'True: {true_label}')


## Cell 9 — Integration with Phase 4 App (app.py snippet)

After training:
1. Download `best_food_classifier.pt` and `food101_labels.json` from Kaggle output
2. Place both in `models/`
3. app.py already has hooks — run `python app/app.py` and the classifier loads automatically

The code below shows exactly what app.py will do when a user uploads a photo.

In [ ]:
"""
This cell demos the integration logic that will be added to app/app.py.
It does NOT import Gradio — just shows the classify_food_crop() function.
"""

import torch.nn.functional as F

def classify_food_crop(
    pil_img: Image.Image,
    mask: np.ndarray,              # binary mask (H, W) float32 from _get_food_mask()
    clf_model: nn.Module,
    class_names: list,
    transform,
    top_k: int = 3,
) -> list:
    """
    Crop the food region from pil_img using the YOLO mask (or full image if mask is thin),
    classify with EfficientNet-B0, return top-k [(label, confidence), …].

    This is the Phase 5 enhancement to app.py's predict() function.
    """
    clf_model.eval()

    # ── Crop bounding box of the mask ─────────────────────────────────────────
    ys, xs = np.where(mask > 0.5)
    if len(ys) > 0:
        y0, y1 = int(ys.min()), int(ys.max())
        x0, x1 = int(xs.min()), int(xs.max())
        # Add 5% padding
        H_mask, W_mask = mask.shape
        pad_y = max(5, int((y1 - y0) * 0.05))
        pad_x = max(5, int((x1 - x0) * 0.05))
        y0 = max(0, y0 - pad_y);  y1 = min(H_mask, y1 + pad_y)
        x0 = max(0, x0 - pad_x);  x1 = min(W_mask, x1 + pad_x)
        crop = pil_img.crop((x0, y0, x1, y1))
    else:
        crop = pil_img   # no mask → use full image

    x = transform(crop.convert('RGB')).unsqueeze(0)  # move to device in app.py
    with torch.no_grad():
        logits = clf_model(x).squeeze(0)
    probs  = F.softmax(logits, dim=0).numpy()
    topk_i = probs.argsort()[-top_k:][::-1]

    return [(class_names[int(i)].replace('_', ' ').title(), float(probs[i])) for i in topk_i]


# ── Demo using a Food-101 validation image ────────────────────────────────────
sample_idx  = random.randint(0, len(val_ds) - 1)
img_path    = val_ds._image_files[sample_idx]
pil_input   = Image.open(str(img_path)).convert('RGB')
true_label  = CLASS_NAMES[val_ds._labels[sample_idx]].replace('_', ' ').title()

W, H   = pil_input.size
centre_mask = np.zeros((H, W), dtype=np.float32)
import cv2 as _cv2
_cv2.circle(centre_mask, (W//2, H//2), int(min(H, W) * 0.4), 1.0, -1)

results = classify_food_crop(pil_input, centre_mask, model, CLASS_NAMES, val_transform, top_k=5)

print(f'True class   : {true_label}')
print('Top-5 predictions:')
for rank, (name, conf) in enumerate(results, 1):
    marker = '✓' if name.lower().replace(' ', '_') == CLASS_NAMES[val_ds._labels[sample_idx]] else ' '
    print(f'  {rank}. {marker} {name:35s}  {conf*100:5.1f}%')

show_prediction(pil_input, title=f'True: {true_label}')


## Cell 10 — Next Steps: Upgrade to More Categories

Once Food-101 is working, upgrade in this order:

### 1. iFood-2019 (251 categories — Kaggle dataset)
```python
# kaggle datasets download -d horizonhardik/ifood-2019-fgvc6
# 251 fine-grained classes, 118K training images
# Change CFG['num_classes'] = 251 and re-run the same training loop
```

### 2. UECFOOD-256 (256 Japanese/Asian food categories)
```python
# Download from: http://foodcam.mobi/dataset256.html
# Best for Asian cuisine: ramen, sushi, gyoza, takoyaki, etc.
```

### 3. Multi-dataset fusion strategy
Merge Food-101 + iFood-2019 + UECFOOD-256 into one dataset:
- Deduplicate overlapping classes (pizza, sushi appear in all three)
- Use a unified label mapping: `{0: 'apple_pie', 1: 'baby_back_ribs', …, N: 'takoyaki'}`
- Result: ~400-500 unique food classes covering Western + Asian cuisine

### 4. Expected accuracy targets
| Dataset | Top-1 (EfficientNet-B0) | Top-5 |
|---|---|---|
| Food-101 | ~82-87% | ~96%+ |
| iFood-2019 | ~75-80% | ~93% |
| Combined | ~72-78% | ~91% |

### Files to download from Kaggle output
After running this notebook:
- `best_food_classifier.pt` — model weights
- `food101_labels.json` — class index → label name mapping
Place both in `models/` alongside `best_mlp.pt`.